# Phase 3 — Factor Validation & GARCH Modeling

**Objective:** Confirm that the 4 primary factors (HML, UMD, RMW, LowVol) capture
distinct risk premia. Fit GARCH volatility models for the tech portfolio.

**Dependencies:** Phase 1 + Phase 2 outputs

**Outputs:**
- `garch_conditional_vol.parquet` — factor-level conditional vol (vol_hml, vol_umd, vol_rmw, vol_lowvol)
- `conditional_vol_series.parquet` — ticker-level daily conditional vol (20 tech stocks)
- `outputs/tables/garch_parameters.csv` — GARCH model selection results per ticker
- Figures: factor cumulative returns, correlation heatmap, GARCH vol plots

In [1]:
# === Setup & Imports ===
import os, sys, warnings, logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats as sp_stats
from arch import arch_model
from tqdm import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Project paths
sys.path.insert(0, str(Path.cwd().parent))
from src.config import (
    PROJECT_ROOT, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR,
    TICKERS, SHORT_HISTORY_TICKERS, COLORS, RANDOM_STATE,
    GARCH_MODELS, GARCH_DISTRIBUTIONS, MIN_OBS_FIGARCH,
    COND_VOL_FILE, GARCH_PARAMS_FILE,
)
from src.validation import validate_parquet, check_nan_propagation, quick_data_check
from src.visualization import setup_style, save_fig, plot_cumulative_returns, plot_correlation_heatmap
from src.garch_utils import (
    fit_garch_family, select_best_garch, extract_conditional_volatility,
    fit_all_tickers_garch, garch_diagnostic_tests,
)
from src.risk_metrics import compute_all_metrics

# Apply consistent plot style
setup_style()
np.random.seed(RANDOM_STATE)

# Logging
PHASE_NUM = 3
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)
logger.info(f"Phase {PHASE_NUM} started")

print(f"Working directory: {PROJECT_ROOT}")

Working directory: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine


---
## Part A: Factor Validation

### A.1 Load Factor Returns

In [2]:
# Load factor_returns_full.parquet (output of Phase 2)
factor_returns = pd.read_parquet(PROCESSED_DIR / 'factor_returns_full.parquet')

validate_parquet(
    factor_returns,
    expected_cols=['hml', 'umd', 'rmw', 'lowvol'],
    min_rows=100,
    date_index=True,
    label='factor_returns_full'
)
quick_data_check(factor_returns, 'Factor Returns Full')

# Keep only the 4 primary factors for this notebook
FACTOR_COLS = ['hml', 'umd', 'rmw', 'lowvol']
factors = factor_returns[FACTOR_COLS].copy()

print(f"Factor data: {factors.shape[0]} months, {factors.shape[1]} factors")
print(f"Date range: {factors.index.min()} to {factors.index.max()}")
factors.head()

=== Factor Returns Full ===
  Shape: (232, 4)
  Index: DatetimeIndex, range: 2005-01-31 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(4)}
  Sorted: True

Factor data: 232 months, 4 factors
Date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00


,hml,umd,rmw,lowvol
date,,,,
2005-01-31,0.0206,0.0296,0.0280,0.024083
2005-02-28,0.0141,0.0343,0.0132,-0.033589
2005-03-31,0.0207,0.0043,0.0044,0.030556
2005-04-30,0.0005,-0.0068,0.0097,0.031510
2005-05-31,-0.0058,0.0037,-0.0102,-0.079741


### A.2 Factor Summary Statistics (Newey-West HAC Standard Errors)

Compute annualised mean, volatility, Sharpe, Sortino, max drawdown, skewness,
kurtosis, and t-statistics using Newey-West HAC standard errors (lag = 4).

In [3]:
# === Factor Summary Statistics ===
summary_rows = []
hac_lag = 4  # Newey-West lag ~ sqrt(T) or 4 for monthly data

for col in FACTOR_COLS:
    series = factors[col].dropna()
    n = len(series)

    # Annualised metrics
    ann_mean = series.mean() * 12
    ann_vol = series.std() * np.sqrt(12)
    sharpe = ann_mean / ann_vol if ann_vol > 0 else 0.0

    # Sortino: downside deviation
    downside = series[series < 0]
    downside_vol = downside.std() * np.sqrt(12) if len(downside) > 0 else ann_vol
    sortino = ann_mean / downside_vol if downside_vol > 0 else 0.0

    # Max drawdown
    cum = (1 + series).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()

    # Higher moments
    skewness = series.skew()
    kurtosis = series.kurtosis()  # excess kurtosis

    # Newey-West HAC t-statistic for mean != 0
    X = np.ones((n, 1))  # intercept-only regression
    model = sm.OLS(series.values, X)
    result = model.fit(cov_type='HAC', cov_kwds={'maxlags': hac_lag})
    t_stat = result.tvalues[0]
    p_value = result.pvalues[0]
    hac_se = result.bse[0]

    summary_rows.append({
        'factor': col.upper(),
        'ann_mean': ann_mean,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_drawdown': max_dd,
        'skewness': skewness,
        'kurtosis': kurtosis,
        'hac_se': hac_se,
        't_stat': t_stat,
        'p_value': p_value,
        'n_obs': n,
    })

factor_summary = pd.DataFrame(summary_rows).set_index('factor')

# Display formatted
print("=" * 80)
print("FACTOR SUMMARY STATISTICS (Newey-West HAC, lag=4)")
print("=" * 80)
display_cols = ['ann_mean', 'ann_vol', 'sharpe', 'sortino', 'max_drawdown',
                'skewness', 'kurtosis', 't_stat', 'p_value']
print(factor_summary[display_cols].round(4).to_string())

# Check: all 4 factors positive mean returns
n_positive = (factor_summary['ann_mean'] > 0).sum()
n_significant = (factor_summary['t_stat'].abs() > 2).sum()
print(f"\nFactors with positive mean: {n_positive}/4")
print(f"Factors with |t| > 2 (HAC): {n_significant}/4 (need >= 3)")

FACTOR SUMMARY STATISTICS (Newey-West HAC, lag=4)
        ann_mean  ann_vol  sharpe  sortino  max_drawdown  skewness  kurtosis  t_stat  p_value
factor                                                                                       
HML      -0.0092   0.1115 -0.0829  -0.1227       -0.5779    0.0348    2.8532 -0.3106   0.7561
UMD       0.0114   0.1546  0.0736   0.0747       -0.5782   -2.3520   15.5768  0.2952   0.7678
RMW       0.0412   0.0644  0.6392   1.2198       -0.1212    0.4876    1.1912  2.7623   0.0057
LOWVOL   -0.1256   0.2009 -0.6251  -0.7690       -0.9465   -1.1037    4.9151 -2.7872   0.0053

Factors with positive mean: 2/4
Factors with |t| > 2 (HAC): 2/4 (need >= 3)


### A.3 LowVol Factor Spanning Regression

Regress LowVol on the other factors (HML, UMD, RMW, and Mkt-RF if available)
to check that LowVol captures an independent risk premium (alpha significant, R-squared < 0.5).

In [4]:
# === LowVol Spanning Regression ===
# Load full factor returns to get Mkt-RF as a control
factor_full = pd.read_parquet(PROCESSED_DIR / 'factor_returns.parquet')

# Align dates
common_idx = factors.index.intersection(factor_full.index)
lowvol_y = factors.loc[common_idx, 'lowvol']

# Build RHS: HML, UMD, RMW, and Mkt-RF (if available)
rhs_cols = []
rhs_df = pd.DataFrame(index=common_idx)

for col in ['hml', 'umd', 'rmw']:
    rhs_df[col] = factors.loc[common_idx, col]
    rhs_cols.append(col)

if 'mkt_rf' in factor_full.columns:
    rhs_df['mkt_rf'] = factor_full.loc[common_idx, 'mkt_rf']
    rhs_cols.append('mkt_rf')

# Drop any rows with NaN
mask = ~(lowvol_y.isna() | rhs_df.isna().any(axis=1))
lowvol_y = lowvol_y[mask]
rhs_df = rhs_df[mask]

X = sm.add_constant(rhs_df)
model = sm.OLS(lowvol_y, X)
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': hac_lag})

print("=" * 70)
print("LowVol Spanning Regression (HAC Standard Errors)")
print("=" * 70)
print(results.summary().tables[1])
print(f"\nR-squared: {results.rsquared:.4f}")
print(f"Adjusted R-squared: {results.rsquared_adj:.4f}")
print(f"Alpha (intercept): {results.params['const']:.6f}")
print(f"Alpha t-stat: {results.tvalues['const']:.4f}")
print(f"Alpha p-value: {results.pvalues['const']:.4f}")
print(f"\nValidation: R-squared < 0.5? {'PASS' if results.rsquared < 0.5 else 'FAIL'}")
print(f"Alpha significant (p < 0.05)? {'YES' if results.pvalues['const'] < 0.05 else 'NO'}")

LowVol Spanning Regression (HAC Standard Errors)
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0089      0.002     -4.203      0.000      -0.013      -0.005
hml           -0.0432      0.067     -0.643      0.520      -0.175       0.089
umd            0.4796      0.105      4.551      0.000       0.273       0.686
rmw            0.8304      0.154      5.407      0.000       0.529       1.131
mkt_rf        -0.6383      0.059    -10.890      0.000      -0.753      -0.523

R-squared: 0.6727
Adjusted R-squared: 0.6670
Alpha (intercept): -0.008859
Alpha t-stat: -4.2028
Alpha p-value: 0.0000

Validation: R-squared < 0.5? FAIL
Alpha significant (p < 0.05)? YES


### A.4 Factor Correlation Matrix

Verify that no factor pair has correlation exceeding 0.6 (low multicollinearity).

In [5]:
# === Correlation Matrix ===
corr_matrix = factors.corr()

print("Factor Correlation Matrix:")
print(corr_matrix.round(4).to_string())

# Check: no pair > 0.6
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
max_corr = upper_tri.max().max()
max_pair = upper_tri.stack().idxmax()
print(f"\nMax pairwise correlation: {max_corr:.4f} ({max_pair[0]}, {max_pair[1]})")
print(f"Validation: No pair > 0.6? {'PASS' if max_corr < 0.6 else 'FAIL'}")

# Correlation heatmap
fig = plot_correlation_heatmap(
    corr_matrix,
    title='Factor Return Correlations',
    figsize=(7, 7)
)
save_fig(fig, 'factor_correlation_heatmap')

Factor Correlation Matrix:
           hml     umd     rmw  lowvol
hml     1.0000 -0.3335  0.0128 -0.2181
umd    -0.3335  1.0000  0.0963  0.5916
rmw     0.0128  0.0963  1.0000  0.3968
lowvol -0.2181  0.5916  0.3968  1.0000

Max pairwise correlation: 0.5916 (umd, lowvol)
Validation: No pair > 0.6? PASS
Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_correlation_heatmap.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_correlation_heatmap.png')

### A.5 Cumulative Return Plots

In [6]:
# === Cumulative Return Plots (factor-coloured) ===
fig, ax = plt.subplots(figsize=(14, 7))

cum_returns = (1 + factors).cumprod()

factor_colors = {
    'hml': COLORS['hml'],
    'umd': COLORS['umd'],
    'rmw': COLORS['rmw'],
    'lowvol': COLORS['lowvol'],
}
factor_labels = {
    'hml': 'HML (Value)',
    'umd': 'UMD (Momentum)',
    'rmw': 'RMW (Quality)',
    'lowvol': 'LowVol',
}

for col in FACTOR_COLS:
    ax.plot(cum_returns.index, cum_returns[col],
            label=factor_labels[col], color=factor_colors[col], linewidth=1.5)

ax.set_ylabel('Cumulative Return (base = 1)')
ax.set_title('Factor Cumulative Returns', fontweight='bold', fontsize=14)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=1, color='grey', linestyle='--', alpha=0.5)

save_fig(fig, 'factor_cumulative_returns')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_cumulative_returns.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_cumulative_returns.png')

---
## Part B: GARCH Pipeline for Tech Portfolio

Fit 4 GARCH models x 4 distributions = up to 16 fits per ticker (320 total).
- Models: GARCH(1,1), GJR-GARCH(1,1,1), EGARCH(1,1), FIGARCH(1,d,1)
- Distributions: Normal, Student's t, Skewed-t, GED
- FIGARCH excluded for tickers with < 1500 observations (CRWD, DDOG, PLTR)

### B.1 Load Tech Portfolio & Compute Daily Log Returns

In [7]:
# === Load master_data.parquet and compute daily log returns ===
master_data = pd.read_parquet(PROCESSED_DIR / 'master_data.parquet')

validate_parquet(
    master_data,
    min_rows=500,
    date_index=True,
    label='master_data'
)
quick_data_check(master_data, 'Master Data')

# Filter to 20 tech tickers only (exclude benchmarks)
available_tickers = [t for t in TICKERS if t in master_data.columns]
missing_tickers = [t for t in TICKERS if t not in master_data.columns]
if missing_tickers:
    print(f"WARNING: Missing tickers: {missing_tickers}")
print(f"Available tickers: {len(available_tickers)}/{len(TICKERS)}")

prices = master_data[available_tickers].copy()

# Compute daily log returns
log_returns = np.log(prices / prices.shift(1)).dropna(how='all')

# Clean: drop any remaining NaN/Inf
log_returns = log_returns.replace([np.inf, -np.inf], np.nan)

print(f"\nLog returns shape: {log_returns.shape}")
print(f"Date range: {log_returns.index.min()} to {log_returns.index.max()}")

# Observation counts per ticker (important for FIGARCH exclusion)
obs_counts = log_returns.count()
print(f"\nObservation counts per ticker:")
for ticker in available_tickers:
    n = obs_counts.get(ticker, 0)
    figarch_ok = "FIGARCH OK" if n >= MIN_OBS_FIGARCH else "FIGARCH EXCLUDED"
    print(f"  {ticker}: {n} obs -- {figarch_ok}")

=== Master Data ===
  Shape: (2556, 27)
  Index: DatetimeIndex, range: 2016-01-04 00:00:00 → 2026-02-27 00:00:00
  Duplicates: 0
  NaN total: 5634 (8.2%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(27)}
  Sorted: True

Available tickers: 20/20

Log returns shape: (2549, 20)
Date range: 2016-01-05 00:00:00 to 2026-02-27 00:00:00

Observation counts per ticker:
  NVDA: 2549 obs -- FIGARCH OK
  AMD: 2549 obs -- FIGARCH OK
  TSM: 2549 obs -- FIGARCH OK
  AVGO: 2549 obs -- FIGARCH OK
  QCOM: 2549 obs -- FIGARCH OK
  MU: 2549 obs -- FIGARCH OK
  AAPL: 2549 obs -- FIGARCH OK
  MSFT: 2549 obs -- FIGARCH OK
  GOOG: 2549 obs -- FIGARCH OK
  META: 2549 obs -- FIGARCH OK
  NFLX: 2549 obs -- FIGARCH OK
  CRM: 2549 obs -- FIGARCH OK
  ADBE: 2549 obs -- FIGARCH OK
  NOW: 2549 obs -- FIGARCH OK
  PANW: 2549 obs -- FIGARCH OK
  CRWD: 1685 obs -- FIGARCH OK
  DDOG: 1616 obs -- FIGARCH OK
  PLTR: 1356 obs -- FIGARCH EXCLUDED
  ANET: 2549 obs -- FIGARCH OK
  XYZ: 0 obs -- FIGARCH EXCLUDED


### B.2 Fit GARCH Models (4 models x 4 distributions x 20 tickers)

**CRITICAL:** `arch` expects percentage returns -- multiply by 100.

In [8]:
# === Fit GARCH family for all tickers ===
# Uses src/garch_utils.py fit_all_tickers_garch which:
#   - Converts decimal returns to percentage (* 100)
#   - Excludes FIGARCH for tickers with < MIN_OBS_FIGARCH observations
#   - Selects best model per ticker by BIC

all_garch_results = fit_all_tickers_garch(log_returns[available_tickers])

print(f"\nGARCH fitting complete for {len(all_garch_results)} tickers")

GARCH fits:   0%|          | 0/20 [00:00<?, ?it/s]

GARCH fits:   5%|▌         | 1/20 [00:02<00:51,  2.71s/it]

GARCH fits:  10%|█         | 2/20 [00:05<00:46,  2.59s/it]

GARCH fits:  15%|█▌        | 3/20 [00:07<00:42,  2.52s/it]

GARCH fits:  20%|██        | 4/20 [00:10<00:40,  2.52s/it]

...

GARCH fits:  85%|████████▌ | 17/20 [00:45<00:06,  2.19s/it]

GARCH fits:  90%|█████████ | 18/20 [00:45<00:03,  1.62s/it]

GARCH fits:  95%|█████████▌| 19/20 [00:48<00:02,  2.02s/it]

GARCH fits: 100%|██████████| 20/20 [00:48<00:00,  2.42s/it]


GARCH fitting complete for 19 tickers


### B.3 Best Model Selection (by BIC)

In [9]:
# === Compile GARCH results table ===
garch_rows = []
best_model_rows = []

for ticker in available_tickers:
    if ticker not in all_garch_results:
        continue

    result = all_garch_results[ticker]
    results_df = result['results_df']
    best = result['best_model']

    # All fits for this ticker
    for _, row in results_df.iterrows():
        garch_rows.append({
            'ticker': ticker,
            'model': row['model'],
            'distribution': row['distribution'],
            'aic': row['aic'],
            'bic': row['bic'],
            'loglikelihood': row['loglikelihood'],
            'converged': row['converged'],
        })

    # Best model
    if best is not None:
        best_model_rows.append({
            'ticker': ticker,
            'best_model': best['model'],
            'best_distribution': best['distribution'],
            'bic': best['bic'],
            'aic': best['aic'],
            'converged': best['converged'],
        })

garch_all_df = pd.DataFrame(garch_rows)
best_models_df = pd.DataFrame(best_model_rows)

# Convergence rate
total_fits = len(garch_all_df)
converged_fits = garch_all_df['converged'].sum()
convergence_rate = converged_fits / total_fits if total_fits > 0 else 0

print("=" * 70)
print("GARCH MODEL SELECTION RESULTS")
print("=" * 70)
print(f"Total fits attempted: {total_fits}")
print(f"Converged fits: {converged_fits} ({convergence_rate:.1%})")
print(f"\nBest model per ticker (by BIC):")
print(best_models_df.to_string(index=False))

# Model type distribution
print(f"\nModel type distribution among best models:")
print(best_models_df['best_model'].value_counts().to_string())
print(f"\nDistribution type among best models:")
print(best_models_df['best_distribution'].value_counts().to_string())

GARCH MODEL SELECTION RESULTS
Total fits attempted: 300
Converged fits: 300 (100.0%)

Best model per ticker (by BIC):
ticker best_model best_distribution          bic          aic  converged
  NVDA     EGARCH         StudentsT 12382.421936 12341.520488       True
   AMD     EGARCH         StudentsT 13335.006379 13294.104931       True
   TSM     EGARCH         StudentsT 10523.603684 10482.702235       True
  AVGO     EGARCH         StudentsT 10971.397120 10930.495672       True
  QCOM     EGARCH         StudentsT 10750.382933 10709.481485       True
    MU     EGARCH         StudentsT 12587.055769 12546.154321       True
  AAPL     EGARCH         StudentsT  9484.871075  9443.969627       True
  MSFT     EGARCH         StudentsT  9076.236277  9035.334829       True
  GOOG     EGARCH             skewt  9549.620581  9502.876069       True
  META     EGARCH         StudentsT 10558.553394 10517.651945       True
  NFLX     EGARCH         StudentsT 11286.498426 11245.596978       True
   CRM

### B.4 Extract Conditional Volatility Series

In [10]:
# === Extract conditional volatility for each ticker ===
cond_vol_dict = {}
for ticker in available_tickers:
    if ticker not in all_garch_results:
        continue
    cond_vol = all_garch_results[ticker]['cond_vol']
    if cond_vol is not None:
        cond_vol_dict[ticker] = cond_vol

cond_vol_df = pd.DataFrame(cond_vol_dict)

# Rename columns to vol_{ticker} convention
cond_vol_df.columns = [f'vol_{t}' for t in cond_vol_df.columns]

print(f"Conditional volatility series shape: {cond_vol_df.shape}")
print(f"Date range: {cond_vol_df.index.min()} to {cond_vol_df.index.max()}")
check_nan_propagation(cond_vol_df, 'Conditional Vol Series')

# Summary statistics of annualised conditional vol
print(f"\nConditional Volatility Summary (annualised decimal):")
vol_summary = cond_vol_df.describe().T[['mean', 'std', 'min', 'max']]
vol_summary.columns = ['Mean Vol', 'Std Vol', 'Min Vol', 'Max Vol']
print(vol_summary.round(4).to_string())

Conditional volatility series shape: (2549, 19)
Date range: 2016-01-05 00:00:00 to 2026-02-27 00:00:00

Conditional Volatility Summary (annualised decimal):
          Mean Vol  Std Vol  Min Vol  Max Vol
vol_NVDA    0.0298   0.0094   0.0151   0.0920
vol_AMD     0.0360   0.0068   0.0245   0.0780
vol_TSM     0.0204   0.0058   0.0091   0.0514
vol_AVGO    0.0228   0.0074   0.0113   0.0887
vol_QCOM    0.0232   0.0075   0.0107   0.0771
vol_MU      0.0299   0.0070   0.0181   0.0713
vol_AAPL    0.0173   0.0067   0.0075   0.0725
vol_MSFT    0.0160   0.0062   0.0067   0.0727
vol_GOOG    0.0178   0.0060   0.0072   0.0587
vol_META    0.0223   0.0082   0.0089   0.0655
vol_NFLX    0.0252   0.0079   0.0141   0.0844
vol_CRM     0.0209   0.0076   0.0083   0.0640
vol_ADBE    0.0206   0.0075   0.0071   0.0739
vol_NOW     0.0246   0.0074   0.0120   0.0680
vol_PANW    0.0235   0.0054   0.0155   0.0761
vol_CRWD    0.0340   0.0091   0.0167   0.0812
vol_DDOG    0.0363   0.0098   0.0198   0.0692
vol_PLTR    0.0

### B.5 GARCH Diagnostic Tests

Run Ljung-Box on standardised residuals, ARCH-LM test, and check persistence
for the best model per ticker.

In [11]:
# === GARCH Diagnostic Tests ===
# Re-fit the best model for each ticker to obtain result objects for diagnostics
diag_rows = []
persistence_values = []

for ticker in tqdm(available_tickers, desc='Diagnostics'):
    if ticker not in all_garch_results:
        continue

    best = all_garch_results[ticker]['best_model']
    if best is None:
        continue

    model_type = best['model']
    dist = best['distribution']
    series = log_returns[ticker].dropna()
    series_pct = series * 100  # CRITICAL: arch expects percentage returns

    # Re-fit to get the result object
    try:
        if model_type == 'GARCH':
            am = arch_model(series_pct, vol='GARCH', p=1, q=1,
                            mean='AR', lags=1, dist=dist)
        elif model_type == 'GJR-GARCH':
            am = arch_model(series_pct, vol='GARCH', p=1, o=1, q=1,
                            mean='AR', lags=1, dist=dist)
        elif model_type == 'EGARCH':
            am = arch_model(series_pct, vol='EGARCH', p=1, o=1, q=1,
                            mean='AR', lags=1, dist=dist)
        elif model_type == 'FIGARCH':
            am = arch_model(series_pct, vol='FIGARCH', p=1, q=1,
                            mean='AR', lags=1, dist=dist)
        else:
            continue

        res = am.fit(disp='off')
        diagnostics = garch_diagnostic_tests(res, lags=10)

        diag_rows.append({
            'ticker': ticker,
            'model': model_type,
            'distribution': dist,
            'ljung_box_resid_pass': diagnostics.get('ljung_box_resid_pass'),
            'ljung_box_sq_pass': diagnostics.get('ljung_box_sq_pass'),
            'arch_lm_pass': diagnostics.get('arch_lm_pass'),
            'persistence': diagnostics.get('persistence'),
            'persistence_valid': diagnostics.get('persistence_valid'),
        })
        persistence_values.append(diagnostics.get('persistence', np.nan))

    except Exception as e:
        print(f"  {ticker}: diagnostic failed -- {e}")
        continue

diag_df = pd.DataFrame(diag_rows)

print("=" * 80)
print("GARCH DIAGNOSTIC TESTS (Best Model per Ticker)")
print("=" * 80)
print(diag_df.to_string(index=False))

# Summary
print(f"\n--- Diagnostic Summary ---")
print(f"Ljung-Box (residuals) pass rate: {diag_df['ljung_box_resid_pass'].mean():.1%}")
print(f"Ljung-Box (squared) pass rate: {diag_df['ljung_box_sq_pass'].mean():.1%}")
print(f"ARCH-LM pass rate: {diag_df['arch_lm_pass'].mean():.1%}")

valid_persistence = diag_df['persistence'].dropna()
if len(valid_persistence) > 0:
    p_mean = valid_persistence.mean()
    p_in_range = ((valid_persistence >= 0.85) & (valid_persistence <= 0.99)).mean()
    print(f"Mean persistence (alpha+beta): {p_mean:.4f}")
    print(f"Persistence in [0.85, 0.99]: {p_in_range:.1%}")

Diagnostics:   0%|          | 0/20 [00:00<?, ?it/s]

Diagnostics:  15%|█▌        | 3/20 [00:00<00:00, 23.83it/s]

Diagnostics:  30%|███       | 6/20 [00:00<00:00, 26.04it/s]

Diagnostics:  45%|████▌     | 9/20 [00:00<00:00, 25.93it/s]

Diagnostics:  60%|██████    | 12/20 [00:00<00:00, 26.40it/s]

Diagnostics:  75%|███████▌  | 15/20 [00:00<00:00, 23.75it/s]

Diagnostics:  95%|█████████▌| 19/20 [00:00<00:00, 26.30it/s]

Diagnostics: 100%|██████████| 20/20 [00:00<00:00, 27.11it/s]

GARCH DIAGNOSTIC TESTS (Best Model per Ticker)
ticker  model distribution  ljung_box_resid_pass  ljung_box_sq_pass  arch_lm_pass  persistence  persistence_valid
  NVDA EGARCH    StudentsT                  True               True          True     1.110952              False
   AMD EGARCH    StudentsT                  True               True          True     1.089211              False
   TSM EGARCH    StudentsT                 False               True          True     1.105157              False
  AVGO EGARCH    StudentsT                  True               True          True     1.109669              False
  QCOM EGARCH    StudentsT                  True               True          True     1.151137              False
    MU EGARCH    StudentsT                 False               True          True     1.085953              False
  AAPL EGARCH    StudentsT                  True               True          True     1.108949              False
  MSFT EGARCH    StudentsT               

### B.6 Factor-Level GARCH Conditional Volatility

Fit GARCH(1,1) to each factor return series to produce the factor conditional
volatility used in later allocation phases.

In [12]:
# === Factor-Level GARCH Conditional Vol ===
# Factor returns are monthly -- convert to percentage for arch
factor_vol_dict = {}

for col in FACTOR_COLS:
    series = factors[col].dropna()
    series_pct = series * 100  # CRITICAL: percentage returns

    # Fit GARCH(1,1) with Student's t (robust to fat tails)
    try:
        am = arch_model(series_pct, vol='GARCH', p=1, q=1,
                        mean='Constant', dist='StudentsT')
        res = am.fit(disp='off')

        # Conditional vol in decimal (divided by 100) -- then annualised by sqrt(12)
        cond_vol = res.conditional_volatility / 100 * np.sqrt(12)
        cond_vol.index = series.index
        factor_vol_dict[f'vol_{col}'] = cond_vol

        # Report parameters
        alpha = res.params.get('alpha[1]', np.nan)
        beta = res.params.get('beta[1]', np.nan)
        persistence = alpha + beta
        print(f"{col.upper()}: alpha={alpha:.4f}, beta={beta:.4f}, "
              f"persistence={persistence:.4f}, converged={res.convergence_flag == 0}")

    except Exception as e:
        # Fallback: rolling volatility
        print(f"{col.upper()}: GARCH failed ({e}), using rolling vol")
        rolling_vol = series.rolling(12).std() * np.sqrt(12)
        factor_vol_dict[f'vol_{col}'] = rolling_vol

factor_cond_vol = pd.DataFrame(factor_vol_dict)
factor_cond_vol = factor_cond_vol.dropna()

print(f"\nFactor conditional vol shape: {factor_cond_vol.shape}")
print(f"Date range: {factor_cond_vol.index.min()} to {factor_cond_vol.index.max()}")
print(f"\nFactor Conditional Vol Summary:")
print(factor_cond_vol.describe().round(4).to_string())

HML: alpha=0.1461, beta=0.8353, persistence=0.9814, converged=True
UMD: alpha=0.3478, beta=0.6192, persistence=0.9670, converged=True
RMW: alpha=0.1107, beta=0.8513, persistence=0.9620, converged=True
LOWVOL: alpha=0.1796, beta=0.7614, persistence=0.9410, converged=True

Factor conditional vol shape: (232, 4)
Date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00

Factor Conditional Vol Summary:
        vol_hml   vol_umd   vol_rmw  vol_lowvol
count  232.0000  232.0000  232.0000    232.0000
mean     0.1066    0.1436    0.0625      0.1949
std      0.0413    0.0806    0.0168      0.0668
min      0.0591    0.0759    0.0420      0.1239
25%      0.0749    0.0977    0.0499      0.1494
50%      0.0876    0.1190    0.0579      0.1768
75%      0.1324    0.1629    0.0692      0.2185
max      0.2194    0.7464    0.1203      0.5639


### B.7 GARCH Conditional Volatility Plots

In [13]:
# === Factor Conditional Volatility Plot ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

for i, col in enumerate(FACTOR_COLS):
    ax = axes[i // 2, i % 2]
    vol_col = f'vol_{col}'
    if vol_col in factor_cond_vol.columns:
        ax.plot(factor_cond_vol.index, factor_cond_vol[vol_col],
                color=factor_colors[col], linewidth=0.8)
        ax.set_title(f'{factor_labels[col]} -- Conditional Volatility', fontweight='bold')
        ax.set_ylabel('Annualised Vol')
        ax.grid(True, alpha=0.3)

plt.suptitle('Factor GARCH(1,1) Conditional Volatility', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, 'factor_garch_conditional_vol')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_garch_conditional_vol.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_garch_conditional_vol.png')

In [14]:
# === Tech Ticker Conditional Volatility Plot (sample of 6) ===
sample_tickers = ['NVDA', 'AAPL', 'MSFT', 'META', 'CRWD', 'TSM']
sample_tickers = [t for t in sample_tickers if f'vol_{t}' in cond_vol_df.columns]

if len(sample_tickers) > 0:
    fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
    axes_flat = axes.flatten()

    for i, ticker in enumerate(sample_tickers[:6]):
        ax = axes_flat[i]
        vol_col = f'vol_{ticker}'
        ax.plot(cond_vol_df.index, cond_vol_df[vol_col],
                color='steelblue', linewidth=0.6, alpha=0.8)
        ax.set_title(f'{ticker} -- GARCH Conditional Volatility', fontweight='bold')
        ax.set_ylabel('Ann. Vol')
        ax.grid(True, alpha=0.3)

    # Hide unused axes
    for j in range(len(sample_tickers), len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle('Tech Portfolio: GARCH Conditional Volatility (Sample)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_fig(fig, 'tech_garch_conditional_vol_sample')
else:
    print("No sample tickers available for plotting")

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/tech_garch_conditional_vol_sample.png


---
## Validation Gates

In [15]:
# === VALIDATION GATES ===
print("=" * 70)
print("PHASE 3 VALIDATION GATES")
print("=" * 70)

gates = {}

# Gate 1: All 4 factors have positive mean returns
n_positive = (factor_summary['ann_mean'] > 0).sum()
gates['All 4 factors positive mean returns'] = n_positive == 4

# Gate 2: >= 3 factors have |t| > 2 (HAC SEs)
n_significant = (factor_summary['t_stat'].abs() > 2).sum()
gates['>= 3 factors t > 2 (HAC SEs)'] = n_significant >= 3

# Gate 3: No factor pair correlation > 0.6
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
max_abs_corr = upper_tri.abs().max().max()
gates['No factor pair corr > 0.6'] = max_abs_corr < 0.6

# Gate 4: LowVol regression R^2 < 0.5
gates['LowVol regression R^2 < 0.5'] = results.rsquared < 0.5

# Gate 5: GARCH convergence >= 90%
gates[f'GARCH convergence >= 90% ({convergence_rate:.1%})'] = convergence_rate >= 0.90

# Gate 6: Persistence in [0.85, 0.99] for most tickers
if len(valid_persistence) > 0:
    p_in_range = ((valid_persistence >= 0.85) & (valid_persistence <= 0.99)).mean()
    gates[f'Persistence in [0.85, 0.99] ({p_in_range:.1%})'] = p_in_range >= 0.5
else:
    gates['Persistence check'] = False

# Gate 7: FIGARCH excluded for CRWD, DDOG, PLTR
short_hist = set(SHORT_HISTORY_TICKERS.keys())
figarch_excluded_properly = True
for ticker in short_hist:
    if ticker in all_garch_results:
        res_df = all_garch_results[ticker]['results_df']
        if 'FIGARCH' in res_df['model'].values:
            figarch_excluded_properly = False
            break
gates['FIGARCH excluded for CRWD/DDOG/PLTR'] = figarch_excluded_properly

for gate_name, passed in gates.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {gate_name}")

n_pass = sum(gates.values())
n_total = len(gates)
print(f"\nResult: {n_pass}/{n_total} gates passed")
logger.info(f"Validation: {n_pass}/{n_total} gates passed")

PHASE 3 VALIDATION GATES
  [FAIL] All 4 factors positive mean returns
  [FAIL] >= 3 factors t > 2 (HAC SEs)
  [PASS] No factor pair corr > 0.6
  [FAIL] LowVol regression R^2 < 0.5
  [PASS] GARCH convergence >= 90% (100.0%)
  [FAIL] Persistence in [0.85, 0.99] (5.3%)
  [FAIL] FIGARCH excluded for CRWD/DDOG/PLTR

Result: 2/7 gates passed


---
## Export Phase 3 Outputs

In [16]:
# === Export all Phase 3 outputs ===

# 1. Factor-level conditional volatility (for factor timing -- Phase 6/7/8)
factor_cond_vol.to_parquet(COND_VOL_FILE)
print(f"Exported garch_conditional_vol.parquet: {factor_cond_vol.shape}")
print(f"  Columns: {list(factor_cond_vol.columns)}")
print(f"  Date range: {factor_cond_vol.index.min()} to {factor_cond_vol.index.max()}")
print(f"  NaN count: {factor_cond_vol.isna().sum().sum()}")

# 2. Ticker-level conditional volatility series (for tech portfolio)
cond_vol_path = PROCESSED_DIR / 'conditional_vol_series.parquet'
cond_vol_df.to_parquet(cond_vol_path)
print(f"\nExported conditional_vol_series.parquet: {cond_vol_df.shape}")
print(f"  Columns: {list(cond_vol_df.columns)[:5]}... ({len(cond_vol_df.columns)} total)")
print(f"  Date range: {cond_vol_df.index.min()} to {cond_vol_df.index.max()}")
print(f"  NaN count: {cond_vol_df.isna().sum().sum()}")

# 3. GARCH parameters table
garch_params_path = GARCH_PARAMS_FILE
# Expand params dict into columns for the best models
garch_export_rows = []
for ticker in available_tickers:
    if ticker not in all_garch_results:
        continue
    best = all_garch_results[ticker]['best_model']
    if best is None:
        continue
    row = {
        'ticker': ticker,
        'model': best['model'],
        'distribution': best['distribution'],
        'bic': best['bic'],
        'aic': best['aic'],
        'converged': best['converged'],
    }
    # Add params from the params dict
    if isinstance(best.get('params'), dict):
        for k, v in best['params'].items():
            row[k] = v
    garch_export_rows.append(row)

garch_params_export = pd.DataFrame(garch_export_rows)
garch_params_export.to_csv(garch_params_path, index=False)
print(f"\nExported garch_parameters.csv: {garch_params_export.shape}")

# 4. Factor summary statistics table
factor_summary.to_csv(TABLES_DIR / 'factor_summary_statistics.csv')
print(f"Exported factor_summary_statistics.csv: {factor_summary.shape}")

logger.info("Phase 3 complete -- all outputs exported")
print("\n=== Phase 3 Complete ===")

Exported garch_conditional_vol.parquet: (232, 4)
  Columns: ['vol_hml', 'vol_umd', 'vol_rmw', 'vol_lowvol']
  Date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00
  NaN count: 0

Exported conditional_vol_series.parquet: (2549, 19)
  Columns: ['vol_NVDA', 'vol_AMD', 'vol_TSM', 'vol_AVGO', 'vol_QCOM']... (19 total)
  Date range: 2016-01-05 00:00:00 to 2026-02-27 00:00:00
  NaN count: 3009

Exported garch_parameters.csv: (19, 33)
Exported factor_summary_statistics.csv: (4, 11)

=== Phase 3 Complete ===
